# Sentence-transformer embeddings (`indonesian-roberta-base-emotion-classifier`) as CatBoost features

In [ ]:
DATA_DIR = "../data"

In [1]:
!pip install transformers sentence-transformers datasets catboost -q

In [2]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

In [3]:
train_df = pd.read_csv(f"{DATA_DIR}/train_cleaned_v2_keywords.csv")
train_df.head()

                                                text             label  \
0  Kunjungan Prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  Anies dapat tepuk tangan meriah saat jadi Rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  Sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  Anies Baswedan Harap aparatur sipil negara ter...           Politik   

   pendidikan rendah  kelompok etnis  masyarakat lintas etnis  kaum difabel  \
0                  0               0                        0             0   
1                  0               0                        0             0   
2                  1               0                        0             0   
3                  0               0                        0             0   
4                  0               0                        0             0   

   bugis  generasi emas  low educated  menengah bawah  ...  nyerocos  \
0      0

In [4]:
train_df_split, val_df_split = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

In [5]:
MODEL = 'StevenLimcorn/indonesian-roberta-base-emotion-classifier'

In [6]:
model = SentenceTransformer(MODEL)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/988 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at StevenLimcorn/indonesian-roberta-base-emotion-classifier and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [13]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [9]:
train_df_split["text"].to_numpy()

array(['Gerakan JNK Bersama Ganjar Mahfud tuh simpel banget tetapi bermakna luas Bersama sama kita bisa bangun dukungan buat Pak Ganjar Pranowo dan Prof Mahfud JNK',
       'Ayo kita bersatu melindungi alam Indonesia seperti Pak Ganjab ngobrol bareng ganjar Ganjar Mahfud 2024 Dulu Jokowi Sekarang Ganjar Coblos 177',
       'Ganjar Pranowo Mahfud sosok Capres Ganjar Bersama Rakyat yang bener bener on point Dukung terus biar makin keren NCT WISH JNK Ganjar Mahfud Rebound Ganjar Pranowo Pilihan Umat',
       ...,
       'Debat harusnya jadi ajang adu gagasan dan memberi wawasan bagi para pemilih Pak Ganjar nampak lebih slow namun setiap jawabannya detail dan substantif Bismillahirrahmanirrahim Ganjar Mahfud 2024 Coblos Nomor 3 riverheaven',
       'Pak Ganjar Pranowo pahlawan petani tembakau Dukung capres yang peduli agar hasil panen tembakau makin melimpah Ganjar Mahfud Rebound Ganjar Pranowo Pilihan Umat JNK',
       'ada yang pernah survey key opinion leader tentang kepemimpinan para c

In [18]:
# max sequence length
max_seq_length = 512
# set max length
texts = []
for text in tqdm(train_df_split["text"].to_numpy()):
  if len(text) > max_seq_length:
    text = text[:max_seq_length]
  texts.append(text)
texts = np.array(texts)

In [19]:
encoded_texts = model.encode(texts)
encoded_texts.shape

(2676, 768)

In [28]:
df = pd.DataFrame(encoded_texts)
df.head()

        0         1         2         3         4         5         6    \
0  0.017594 -0.224640 -0.168418  1.547908 -1.814274  0.853116 -1.210886   
1 -0.217106 -0.121568  0.187908  0.911719 -0.191763  0.598863 -1.411355   
2  0.083448 -0.700708  0.104572  0.997051 -0.852737  1.473669 -1.555106   
3  0.322775 -0.738688  0.581838  1.599612 -0.488952  1.032307 -1.103573   
4  0.595147 -0.324096 -0.353024  1.919566 -0.215094  1.294995 -1.485762   

        7         8         9    ...       758       759       760       761  \
0 -0.504165 -0.426381  0.704657  ...  0.365193 -0.728897 -0.695749  0.168640   
1 -1.146405 -1.594965  0.830880  ... -0.246765 -0.715356 -1.718343 -0.047849   
2 -1.005122  0.319123  1.270799  ... -0.144349 -0.388732 -0.715617  0.631733   
3 -0.349832  0.073829  0.805977  ...  0.294534 -0.997078 -1.470733  0.783037   
4 -0.014105  0.173584  1.279907  ... -0.358153 -0.648237 -0.649640  0.842952   

        762       763       764       765       766       767  
0  0

In [25]:
# make df with embedding column, and inside that column is a list
# df = pd.DataFrame({'embedding':list(encoded_texts)})
# df.head()

                                           embedding
0  [0.017593767, -0.22464006, -0.16841759, 1.5479...
1  [-0.21710643, -0.12156841, 0.18790781, 0.91171...
2  [0.083448246, -0.70070815, 0.10457184, 0.99705...
3  [0.32277483, -0.738688, 0.5818384, 1.5996125, ...
4  [0.5951467, -0.32409593, -0.35302353, 1.919565...

In [29]:
df.to_csv(f"{DATA_DIR}/train_embeddings.csv", index=False)

In [7]:
text_embeddings_df = pd.read_csv(f"{DATA_DIR}/train_embeddings.csv")
# text_embeddings_df = pd.DataFrame({'embedding':list(text_embeddings_df)})
text_embeddings_df.head()

          0         1         2         3         4         5         6  \
0  0.017594 -0.224640 -0.168418  1.547908 -1.814274  0.853116 -1.210886   
1 -0.217106 -0.121568  0.187908  0.911719 -0.191763  0.598863 -1.411355   
2  0.083448 -0.700708  0.104572  0.997051 -0.852737  1.473669 -1.555106   
3  0.322775 -0.738688  0.581838  1.599613 -0.488952  1.032307 -1.103573   
4  0.595147 -0.324096 -0.353024  1.919566 -0.215094  1.294995 -1.485762   

          7         8         9  ...       758       759       760       761  \
0 -0.504165 -0.426381  0.704657  ...  0.365193 -0.728897 -0.695749  0.168640   
1 -1.146405 -1.594966  0.830880  ... -0.246765 -0.715356 -1.718343 -0.047849   
2 -1.005122  0.319123  1.270799  ... -0.144349 -0.388732 -0.715617  0.631733   
3 -0.349832  0.073829  0.805977  ...  0.294534 -0.997078 -1.470733  0.783037   
4 -0.014105  0.173584  1.279907  ... -0.358153 -0.648237 -0.649640  0.842952   

        762       763       764       765       766       767  
0  0

In [8]:
train_df = pd.concat([train_df, text_embeddings_df], axis=1)
train_df.head()

                                                text             label  \
0  Kunjungan Prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  Anies dapat tepuk tangan meriah saat jadi Rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  Sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  Anies Baswedan Harap aparatur sipil negara ter...           Politik   

   pendidikan rendah  kelompok etnis  masyarakat lintas etnis  kaum difabel  \
0                  0               0                        0             0   
1                  0               0                        0             0   
2                  1               0                        0             0   
3                  0               0                        0             0   
4                  0               0                        0             0   

   bugis  generasi emas  low educated  menengah bawah  ...       758  \
0      0

In [9]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

In [10]:
train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

train_pool = Pool(
    # data=train_df_split[kw_df["keyword"].to_list() + ["text", "embedding"]].to_numpy(),
    data=train_df_split.drop("label", axis=1).to_numpy(),
    label=train_df_split['label'].to_numpy(),
    text_features=['text'],
    # embedding_features=["embedding"],
    # feature_names=kw_df["keyword"].to_list() + ["text", "embedding"]
    feature_names=train_df_split.drop("label", axis=1).columns.to_list()
)

val_pool = Pool(
    # data=val_df_split[kw_df["keyword"].to_list() + ["text", "embedding"]].to_numpy(),
    data=val_df_split.drop("label", axis=1).to_numpy(),
    label=val_df_split['label'].to_numpy(),
    text_features=['text'],
    # embedding_features=["embedding"],
    # feature_names=kw_df["keyword"].to_list() + ["text", "embedding"]
    feature_names=train_df_split.drop("label", axis=1).columns.to_list()
)

In [11]:
class_weights_dict = dict()
n, num_class = train_df_split.shape[0], len(train_df_split['label'].unique())
for label in train_df_split['label'].unique():
    n_l = train_df_split[train_df_split['label'] == label].shape[0]
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

{'Ekonomi': 1.8583333333333334,
 'Politik': 0.18745796906523202,
 'Pertahanan dan Keamanan': 2.0496323529411766,
 'Sosial Budaya': 1.39375,
 'Ideologi': 1.883445945945946,
 'Sumber Daya Alam': 4.35546875,
 'Demografi': 8.198529411764707,
 'Geografi': 25.34090909090909}

In [12]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = class_weights_dict
)

cb_model.fit(
    train_pool,
    eval_set=val_pool
)

Learning rate set to 0.114028
0:	learn: 0.3270930	test: 0.2345055	best: 0.2345055 (0)	total: 91ms	remaining: 1m 30s
100:	learn: 0.7962859	test: 0.3715932	best: 0.4750349 (34)	total: 3.32s	remaining: 29.6s
200:	learn: 0.8680305	test: 0.3631615	best: 0.4750349 (34)	total: 6.26s	remaining: 24.9s
300:	learn: 0.8971763	test: 0.3490316	best: 0.4750349 (34)	total: 11.3s	remaining: 26.3s
400:	learn: 0.9184934	test: 0.3340547	best: 0.4750349 (34)	total: 14.2s	remaining: 21.2s
500:	learn: 0.9391948	test: 0.3333447	best: 0.4750349 (34)	total: 17.5s	remaining: 17.5s
600:	learn: 0.9494926	test: 0.3357200	best: 0.4750349 (34)	total: 26.5s	remaining: 17.6s
700:	learn: 0.9579522	test: 0.3326747	best: 0.4750349 (34)	total: 29.3s	remaining: 12.5s
800:	learn: 0.9632219	test: 0.3317608	best: 0.4750349 (34)	total: 32.5s	remaining: 8.07s
900:	learn: 0.9684118	test: 0.3333041	best: 0.4750349 (34)	total: 41.1s	remaining: 4.52s
999:	learn: 0.9717108	test: 0.3345472	best: 0.4750349 (34)	total: 43.9s	remaining: 

In [13]:
# Text + Binary + Embeddings
# Make predictions on the validation set
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.48158112987275703
